# CHRASS A/B Probe (Colab)

Roda 8 modelos gêmeos (densities 0, 10, 20, 30, 40, 50, 60, 70%) com mesma seed, dados, lr.
Compara loss curve + final eval pra decidir se CHRASS vai ligado no bundle do amigo.

**Hardware esperado:** Colab T4 (free tier). Total ~4-6h.
**Modelo:** 6 layers × d=192 × 4 heads × vocab=8192 ≈ 5M params.

Para 40M, mude `LAYERS=12 D_MODEL=384` na célula 3.

## 1. Verificar GPU + dependências

In [ ]:
!nvidia-smi | head -15
!pip install --quiet pybind11 numpy zstandard matplotlib

## 2. Clonar repo + build nsos_ext

**Você deve ter acesso ao repo `reimagined-main`** (private). Substitua a URL.
Build CUDA: ~10 min em T4.

In [ ]:
# !git clone <YOUR_REPO_URL> reimagined-main
# Ou: subir o .zip via Colab UI e descompactar
%cd /content
import os
if not os.path.isdir('/content/reimagined-main'):
    raise RuntimeError('Suba o repo manualmente para /content/reimagined-main')
%cd /content/reimagined-main
!pwd
!ls OXN/nsos/include/chrass_layer_v2.h

In [ ]:
# Build CUDA-enabled (T4 = sm_75)
!mkdir -p OXN/nsos/build-colab
!cmake -S OXN/nsos -B OXN/nsos/build-colab \
    -DNSOS_ENABLE_CUDA=ON \
    -DNSOS_BUILD_PYTHON=ON \
    -DNSOS_BUILD_TESTS=OFF \
    -DNSOS_BUILD_CLI=OFF \
    -DNSOS_BUILD_API=OFF \
    -DNSOS_BUILD_OXTAMEM=OFF \
    -DNSOS_CUDA_ARCHITECTURES=75 \
    2>&1 | tail -5
!cmake --build OXN/nsos/build-colab --target nsos_ext -j 2>&1 | tail -5

In [ ]:
# Validar import
import sys
sys.path.insert(0, '/content/reimagined-main/OXN/nsos/build-colab')
import nsos_ext
print('nsos_ext imported OK. ModelConfig has use_chrass:', hasattr(nsos_ext.ModelConfig(), 'use_chrass'))

## 3. Configurar e rodar 8 densities

In [ ]:
# Parâmetros do sweep
LAYERS = 6        # 6 layers ≈ 5M params; mude para 12 para ~40M
D_MODEL = 192     # 192 com 6 layers ≈ 5M; com 12 layers e d=384 ≈ 40M
N_HEADS = 4
VOCAB = 8192
STEPS = 2000      # ~30 min/run em T4 pra config 5M
SEED = 42         # MESMA seed pra todos -> A/B controlado
DENSITIES = [0, 10, 20, 30, 40, 50, 60, 70]

In [ ]:
# Subir dados (opcional). Sem isso, usa corpus sintético PT contábil embutido.
# Pra usar real: upload um shard tipo /content/data/corpus_a_0000.jsonl.zst
DATA_PATH = ''  # '' = sintético; ou path absoluto para .jsonl(.zst)

In [ ]:
import subprocess
import time

out_dir = '/content/probe_reports'
!mkdir -p {out_dir}

for d in DENSITIES:
    print(f'\n{"="*60}\nDensity {d}% — starting\n{"="*60}')
    t0 = time.time()
    cmd = [
        'python', '/content/reimagined-main/OXN/nsos/scripts/probe_chrass_ab.py',
        '--density', str(d),
        '--seed', str(SEED),
        '--steps', str(STEPS),
        '--layers', str(LAYERS),
        '--d-model', str(D_MODEL),
        '--n-heads', str(N_HEADS),
        '--vocab-size', str(VOCAB),
        '--out-dir', out_dir,
        '--cuda',
    ]
    if DATA_PATH:
        cmd += ['--data-path', DATA_PATH]
    env = {**os.environ, 'PYTHONPATH': '/content/reimagined-main/OXN/nsos/build-colab'}
    p = subprocess.run(cmd, env=env, capture_output=True, text=True)
    print(p.stdout[-3000:])
    if p.returncode != 0:
        print('STDERR:', p.stderr[-1500:])
    print(f'   density {d}% done in {(time.time()-t0)/60:.1f} min')

## 4. Analisar resultados

In [ ]:
!python /content/reimagined-main/OXN/nsos/scripts/analyze_chrass_ab.py \
    --reports-dir /content/probe_reports \
    --plot

In [ ]:
# Visualizar plot inline
from IPython.display import Image, display
display(Image('/content/probe_reports/chrass_ab_curves.png'))

In [ ]:
# Empacotar reports pra download
!cd /content && zip -r probe_reports.zip probe_reports/
from google.colab import files
files.download('/content/probe_reports.zip')